# Finds ply files and loads/saves to update element and property names.

In [2]:
from pathlib import Path
import os
from pymathutils.mesh import load_mesh_samples, save_mesh_samples

ply_dir_in = Path("../data/example_ply")
ply_dir_out = Path("../data/example_ply_feb_29_26")
os.system(f"mkdir -p {ply_dir_out}")
    

for ply_path in ply_dir_in.rglob("*.ply"):
    print(f"{ply_path} -> {ply_dir_out}/{ply_path.name}")
    # s = load_mesh_samples(str(ply_path))
    s = load_mesh_samples(str(ply_path), ply_property_convention="MeshBrane")
    # print(s.keys())
    s["X_ambient_V"] = s.pop("xyz_coord_V")
    # break
    save_mesh_samples(s, f"{ply_dir_out}/{ply_path.name}")
    


../data/example_ply/fibonacci_sphere/sphere_0064000_he.ply -> ../data/example_ply_feb_28_26/fibonacci_sphere/sphere_0064000_he.ply
../data/example_ply/fibonacci_sphere/sphere_0016000_he.ply -> ../data/example_ply_feb_28_26/fibonacci_sphere/sphere_0016000_he.ply
../data/example_ply/fibonacci_sphere/sphere_0000500_he.ply -> ../data/example_ply_feb_28_26/fibonacci_sphere/sphere_0000500_he.ply
../data/example_ply/fibonacci_sphere/sphere_0256000_he.ply -> ../data/example_ply_feb_28_26/fibonacci_sphere/sphere_0256000_he.ply
../data/example_ply/fibonacci_sphere/sphere_0008000_he.ply -> ../data/example_ply_feb_28_26/fibonacci_sphere/sphere_0008000_he.ply
../data/example_ply/fibonacci_sphere/sphere_0002000_he.ply -> ../data/example_ply_feb_28_26/fibonacci_sphere/sphere_0002000_he.ply


In [10]:
from pathlib import Path
import os
from pymathutils.mesh import load_mesh_samples, save_mesh_samples,tri_cycles_to_half_edge_samples, tri_cycles_to_edge_cycles, half_edge_samples_no_edge_data_to_edge_tri_cycles

ply_dir_in = Path("../data/example_ply_old")
ply_dir_out = Path("../data/example_ply_feb_29_26")

os.system(f"mkdir -p {ply_dir_out}")

def re_ply(ply_paths):
    he_keys = set(["f_left_H", "h_next_H", "h_out_V", "h_right_F", "h_twin_H", "v_origin_H", "h_directed_E", "e_undirected_H"]) # plus h_negative_B
    non_he_keys = set(["xyz_coord_V", "X_ambient_V", "V_cycle_E", "V_cycle_F"])
    s_keys = set(["V_cycle_E", "V_cycle_F"])
    
    def ensure_EF_cycles(s):
        if "V_cycle_F" in s.keys():
            if "V_cycle_E" not in s.keys():
                s["V_cycle_E"] = tri_cycles_to_edge_cycles(s["V_cycle_F"]) # vf to here
            return
        he_samples =  {k: v for k,v in s.items() if (k not in non_he_keys)}
        et_samples = half_edge_samples_no_edge_data_to_edge_tri_cycles(he_samples)
        s |= et_samples # he to here
    
    def ensure_he_data(s):
        # assumes V_cycle_F is present
        if he_keys <= s.keys():
            print("he_keys present")
            return
        s |= tri_cycles_to_half_edge_samples(s["V_cycle_F"])
        
        
        
    for ply_path in ply_paths:
        print(f"{ply_path} -> {ply_dir_out}/{ply_path.name}")
        s = load_mesh_samples(str(ply_path), ply_property_convention="MeshBrane")
        s["X_ambient_V"] = s.pop("xyz_coord_V")
        # print("load_mesh_samples")
        # for k,v in s.items():
        #     print(f"{k}: {v.shape}")
        ensure_EF_cycles(s)
        # print("ensure_EF_cycles")
        # for k,v in s.items():
        #     print(f"{k}: {v.shape}")
        ensure_he_data(s)
        # print("ensure_he_data")
        # for k,v in s.items():
        #     print(f"{k}: {v.shape}")
        save_mesh_samples(s, f"{ply_dir_out}/{ply_path.name}", use_binary=False)

def reply_all_recursive():
    ply_paths = ply_dir_in.rglob("*.ply")
    re_ply(ply_paths)

def reply_all():
    ply_paths = ply_dir_in.glob("*.ply")
    re_ply(ply_paths)


def reply_few():
    # ply_paths = ply_dir_in.glob("*.ply")
    ply_names = ["annulus_he.ply",  "dumbbell_coarse_he.ply"]
    ply_paths = [ply_dir_in / name for name in ply_names]
    re_ply(ply_paths)

reply_few()
# reply_all()

../data/example_ply_old/annulus_he.ply -> ../data/example_ply_feb_29_26/annulus_he.ply
../data/example_ply_old/dumbbell_coarse_he.ply -> ../data/example_ply_feb_29_26/dumbbell_coarse_he.ply
